# 次のステップ整理：木材近赤外スペクトルによる含水率予測

作成: 2026-04-17  
目的: 現状を踏まえ、何をどの順番で試すべきかを整理する

---

## 現状スコアまとめ

| バージョン | 前処理 | 特徴量 | モデル | CV方式 | 提出RMSE |
|---|---|---|---|---|---|
| v1 | SNV + PCA(50) | PCA50 + species番号 | LightGBM | sample単位（リーク有） | 20.062 |
| **v2** | SNV + PCA(50) | PCA50 + species番号 | LightGBM | **樹種単位（正しい）** | **18.092** |
| v3 | SNV + SG2次微分 | 生スペクトル全波数 | PLSR | 樹種単位 | 23.037 |

→ 現時点のベストは **v2 (RMSE=18.092)**

---

## 最重要課題：ゼロショット汎化

**trainとtestで樹種が完全に異なる（共通樹種ゼロ）**

- train: イチョウ・ウエンジ・ウォールナット・クリ・スプルース・チェリー・トチ・ナラ・ヒノキ・ベイスギ・ベイマツ・ホワイトオーク・米ヒバ（13種）
- test: クスノキ・ケヤキ・スギ・タモ・チーク・ヤマザクラ（6種）

**これが本コンペの本質的な難しさ。**  
「樹種固有の特性」ではなく「水の吸収」というドメイン知識を活かした特徴量・モデル設計が必要。

### CV設計の正解
- `GroupKFold(groups=樹種)` で test-like な評価をする
- sample単位の分割はリークになるので NG
- ベイスギはCVのfoldから除外（汎化を妨げる可能性。v2で検証済み）

---

## 前処理の詳細

### 入力データの構造

1スペクトル = 1サンプル = 1行。列は波数（cm⁻¹）で、値は吸光度（absorbance）。

```
吸光度 = -log10(試料反射光 / リファレンス白色板反射光)
```

値が大きいほど「光を吸収している（その波数のエネルギーを使っている）」。  
含水率が高いほど水が多く、水の吸収ピーク付近の吸光度が上がる。

---

### ノイズ源（なぜ前処理が必要か）

| ノイズの種類 | 具体的な現象 | 前処理 |
|---|---|---|
| **散乱** | 木材表面の粗さ・粒径の違いでスペクトル全体が縦にスケール変化 | SNV / MSC |
| **ベースラインのオフセット** | スペクトル全体が一定量だけ上下にずれる | SG微分 |
| **ベースラインの傾き** | 高波数側と低波数側で吸光度が傾く | SG微分 |
| **プローブ距離変化** | 乾燥収縮で試料がプローブから少し遠ざかる → スケール変化 | SNV / MSC |

散乱・光路長変動は「**スペクトル全体を同じ割合で引き伸ばす**」ような効果なので、  
各ピーク間の**相対的な形状**は保たれる。SNVはこの相対形状だけを残す処理。

---

### Step 1: SNV（Standard Normal Variate）

**やっていること**: 各サンプル（行）単位で、平均0・標準偏差1になるよう標準化する。

```
SNV後の吸光度[i, j] = (吸光度[i, j] - 吸光度[i, :].mean()) / 吸光度[i, :].std()
```

- `i` = サンプルのインデックス（行）
- `j` = 波数のインデックス（列）
- **行単位** の標準化（列単位ではない）

**なぜ効くか**: 散乱が強いと全体が底上げ/引き伸ばされるが、  
平均を引いて標準偏差で割ると、絶対値のスケールに依存しなくなる。  
「どの波数で、どれだけ他の波数より吸光度が高いか」という形状情報だけが残る。

**注意点**: 標準偏差が0のフラットなスペクトル（異常スペクトル）で除算エラーになるため `+1e-8` でガード。

```python
def snv(X):
    mean = X.mean(axis=1, keepdims=True)  # 行ごとの平均
    std  = X.std(axis=1, keepdims=True)   # 行ごとの標準偏差
    return (X - mean) / (std + 1e-8)
```

**使う場面**: 全手法で先頭に必ず適用する。

---

### Step 2: SG 2次微分（Savitzky-Golay Derivative）

**やっていること**: SNV後のスペクトルに対して「平滑化しながら2次微分」を取る。

**2次微分とは何か**:  
スペクトルの曲率（曲がり具合）を見る。ピーク頂点では2次微分が負の最小値になる。

```
生スペクトル（含水率グラデーション）
  ↓ 2次微分
ピーク位置が「負のスパイク」になる。ベースラインはゼロに近くなる。
```

**なぜ Savitzky-Golay か**: 普通の有限差分で微分するとノイズを増幅してしまう。  
SG法は局所的に多項式をフィットしてから微分するため、**ピーク形状を保ちながらスムーズに微分**できる。

```python
from scipy.signal import savgol_filter

# X.shape = (n_samples, n_wavenumbers)
X_sg = savgol_filter(X, window_length=11, polyorder=2, deriv=2, axis=1)
```

- `window_length=11`: 前後11点（約88 cm⁻¹）を使って局所フィット
- `polyorder=2`: 2次多項式でフィット
- `deriv=2`: 2次微分を出力

**windowサイズのトレードオフ**:
- 小さい（例 5）→ 微分が鋭い。ノイズも鋭くなる
- 大きい（例 21）→ 平滑化が強い。微分は鈍くなるがノイズに強い
- 現在 window=11 → チューニング余地あり

**SG後の値のスケール**: 吸光度の「変化率の変化率」なので値は非常に小さくなる。  
そのためSGの後にさらにSNVを一度かけてもよい（実験の価値あり）。

**使う場面**: PLSRに投入する前に適用。LightGBMにはPCA後のため現状は使っていない。

---

### Step 3: MSC（Multiplicative Scatter Correction）【未実装】

**やっていること**: 各サンプルを「trainの平均スペクトル」に線形回帰フィットして、  
散乱成分（比例係数・オフセット）を除去する。

```
吸光度[i, :] ≈ a[i] × 平均スペクトル + b[i]
→ MSC補正後 = (吸光度[i, :] - b[i]) / a[i]
```

SNVとの違い: SNVは「そのサンプル自身の平均・標準偏差」で正規化するが、  
MSCは「train全体の平均スペクトル（基準）」に合わせて正規化する。  
基準があるぶん、スペクトルの形状情報をより保持できるとされる。

**ルール上の注意**: trainの平均スペクトルをfitするのはtrainのみで行い、  
testは「1サンプルずつ」trainの平均スペクトルで補正する → ルールOK。

```python
ref = X_train.mean(axis=0)  # trainのみでfit
for i in range(len(X_test)):
    coef = np.polyfit(ref, X_test[i], 1)       # a, b を推定
    X_test_msc[i] = (X_test[i] - coef[1]) / coef[0]  # 補正
```

---

### 前処理まとめ（比較表）

| 処理 | 何を除去するか | 実装 | 推奨用途 |
|---|---|---|---|
| SNV | 散乱・光路長変動（スケール） | ✅済 | 全手法の先頭 |
| SG 2次微分 | ベースラインのオフセット・傾き | ✅済 | PLSR入力 |
| MSC | 散乱（SNVより精密） | ❌未 | SNVの代替・比較 |
| 波数帯域絞り込み | 含水率と無関係なノイズ波数 | ❌未 | 手作り特徴量の前 |

---

## 特徴量エンジニアリング方針

### 方針：「1本の木から物理的に意味のある数値を手で作る」

現在のPCA50次元は「統計的な主成分」であり物理的意味がない。  
水の吸収帯を中心に**人間が解釈できる特徴量**を作ることで、樹種をまたいだ汎化を狙う。

---

### 水の吸収帯（対象領域）

```
10000  9000  8000  7000  6000  5000  4000  cm⁻¹
  |           |     |          |
              8500  6900       5200
              ↑     ↑          ↑
           第2倍音  第1倍音    結合音（OH伸縮+変角）
```

5200 cm⁻¹ 帯は「二重ピーク構造（ダブレット）」を持つことがある。  
→ 「2つのピーク間隔」という特徴量が有効になる可能性。

---

### 特徴量アイデア一覧

#### A. 帯域ごとの統計量（各水ピーク帯で計算する）

| 特徴量名 | 計算 | 意味 |
|---|---|---|
| `peak_max_{band}` | 帯域内の吸光度最大値 | ピーク高さ |
| `peak_min_{band}` | 帯域内の吸光度最小値 | ベース吸光度 |
| `peak_range_{band}` | max - min | ピークの「立ち上がり幅」。含水率が高いほど大きくなるはず |
| `peak_mean_{band}` | 帯域内の平均吸光度 | 帯域全体の水吸収量 |
| `peak_area_{band}` | 帯域内の吸光度の和（台形積分） | より精密な吸収量 |
| `peak_pos_{band}` | 最大吸光度を示す波数 | ピーク位置のシフト（含水状態で動く可能性） |

#### B. ピーク間の関係量（帯域をまたぐ）

| 特徴量名 | 計算 | 意味 |
|---|---|---|
| `ratio_6900_5200` | peak_area_6900 / peak_area_5200 | 2つの水ピークの比率。含水形態（自由水vs結合水）を反映する可能性 |
| `ratio_8500_6900` | peak_area_8500 / peak_area_6900 | 〃 |
| `pos_diff_6900_5200` | peak_pos_6900 - peak_pos_5200 | 2ピーク間の波数距離 |

#### C. 5200 cm⁻¹ 帯のダブレット（2山構造）

5200 cm⁻¹ 付近では含水率が高いと2山が出やすい（OH伸縮と変角の結合音が分離して見える）。

| 特徴量名 | 計算 | 意味 |
|---|---|---|
| `doublet_gap` | 5200帯内の2つのローカルピーク間の波数差 | 2山の間隔 |
| `doublet_ratio` | 2山の高さの比 | 左右のバランス |
| `doublet_valley` | 2山の間の谷の深さ | 2山がどれだけ分離しているか |

#### D. スペクトル全体の形状

| 特徴量名 | 計算 | 意味 |
|---|---|---|
| `global_max` | 全波数での吸光度最大値の波数 | 最も強い吸収帯がどこか |
| `global_range` | 全波数での max - min | スペクトル全体のダイナミックレンジ |
| `slope_high_low` | 高波数側の平均 - 低波数側の平均 | スペクトルの傾き |

---

### 実装の進め方

1. SNVを適用した後のスペクトルを使う
2. 各帯域でマスクを作って統計量を計算
3. ダブレット検出には `scipy.signal.find_peaks` を使う
4. 作った特徴量をLightGBMに投入してCV評価
5. feature importanceで効いている特徴量を確認

**注意**: PLSRには生スペクトルをそのまま入れる方が向いている。  
手作り特徴量はLightGBMとの組み合わせで使う。

---

## モデル方針

### 現在のモデル
- **LightGBM** (v1/v2): 汎用的な勾配ブースティング。PCA50を入力
- **PLSR** (v3): スペクトル解析の王道。全波数を直接入力

### LightGBM
- 改善できることは →
  - `num_leaves`, `learning_rate`, `min_child_samples` のチューニング（optuna推奨）
  - species番号を除外、または embedding特徴量に置き換え
  - 波数帯域絞り込み後のPCAを入力にする

### PLSR（Partial Least Squares Regression）
- スペクトル → 含水率の直接モデリングに強い
- v3のスコア(23.037)が悪かった原因: SG+PLSRの組み合わせ、またはn_componentsの設定
- 改善できることは →
  - **SNV+PCAなしの生スペクトルをPLSRに直接投入**して試す
  - n_componentsをCVで再チューニング（現在のCVが正しく機能しているか要確認）
  - SG微分ありとなしで比較

### 新規モデル候補①: Ridge / ElasticNet
- スペクトルデータ（高次元・共線性あり）に強い
- PLSRより実装が単純で解釈しやすい
- 改善できることは → SNV後の全波数に直接Ridgeをかける（PCA不要）

### 新規モデル候補②: SVR（Support Vector Regression）
- 少ないサンプルでも機能しやすい
- kernel='rbf' か kernel='linear' を試す

### アンサンブル戦略【最優先で試す価値あり】
```
最終予測 = α × LightGBM予測 + (1-α) × PLSR予測
```
- v2(LGBM=18.092)とv3(PLSR=23.037)は手法が全く異なるため、誤差が相補的な可能性
- 改善できることは → αをCVで最適化（0.1刻みで探索）
- まずα=0.7（LGBM重め）から試す

---

## 優先実施リスト（上から順に試す）

### 🔴 即効性高・実装コスト低

1. **PLSRの再チューニング**  
   - SNV後の生スペクトルをそのままPLSRに入れてCV評価
   - n_componentsを5〜30で探索  
   - → v3が23.037だった原因を特定する

2. **LightGBM + PLSRのアンサンブル**  
   - v2の予測とPLSR(再チューニング後)の予測を混ぜる  
   - `pred = 0.7 * pred_lgbm + 0.3 * pred_plsr` から試す

3. **波数帯域の絞り込み**  
   - 8300〜8700, 6700〜7100, 5000〜5400 cm⁻¹ の3帯域に限定  
   - SNV後にこの帯域だけ抽出してPLSR/LightGBMに投入

4. **SGのwindowチューニング**  
   - window = [5, 7, 11, 15, 21] でCVを回してベストを選ぶ

### 🟡 効果不明・実装コスト中

5. **MSCの追加**  
   - trainの平均スペクトルでfitし、train/testそれぞれ変換  
   - SNV+MSCとSNV単独を比較

6. **Ridge/ElasticNet**  
   - SNV後の全波数に対して直接学習（PCA不要）  
   - alphaをCV最適化

7. **ベイスギ以外の除外候補探索**  
   - 樹種別OOF残差を確認し、外れ値的な樹種を特定  
   - 除外してCVが改善するか確認

### 🟢 高難度・高リターン（余裕があれば）

8. **樹種名のテキストembedding化**  
   - `sentence-transformers` の多言語モデルを使用  
   - 樹種embedding + スペクトルPCAをLightGBMに投入  
   - testの6樹種に汎化できるか確認

---

## 手作り特徴量の実装

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks
import matplotlib
matplotlib.rcParams['font.family'] = 'MS Gothic'

# --- データ読み込み ---
train = pd.read_csv("../data/raw/train.csv", encoding="shift-jis")
test  = pd.read_csv("../data/raw/test.csv",  encoding="shift-jis")

META_COLS   = ["sample number", "species number", "樹種", "含水率"]
wave_cols   = [c for c in train.columns if c not in META_COLS]
wavenumbers = np.array(wave_cols, dtype=float)

X_train_raw = train[wave_cols].values.astype(float)
y_train     = train["含水率"].values
groups      = train["樹種"].values
X_test_raw  = test[wave_cols].values.astype(float)

def snv(X):
    mean = X.mean(axis=1, keepdims=True)
    std  = X.std(axis=1, keepdims=True)
    return (X - mean) / (std + 1e-8)

X_snv_train = snv(X_train_raw)
X_snv_test  = snv(X_test_raw)

print(f"train: {X_snv_train.shape}, test: {X_snv_test.shape}")
print(f"波数範囲: {wavenumbers.min():.0f} ~ {wavenumbers.max():.0f} cm⁻¹")

In [ ]:
# --- 水ピーク帯域の可視化（どの帯域を使うか確認する）---
# SNV後の平均スペクトルで水ピーク帯域の位置を確認

mean_snv = X_snv_train.mean(axis=0)

# 含水率でグループ分けして帯域の変化を見る
low_mc  = X_snv_train[y_train <  20]   # 低含水率
high_mc = X_snv_train[y_train > 100]   # 高含水率

BANDS = {
    "8500帯 (第2倍音)": (8300, 8700),
    "6900帯 (第1倍音)": (6700, 7100),
    "5200帯 (結合音)":  (5000, 5400),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (band_name, (lo, hi)) in zip(axes, BANDS.items()):
    mask = (wavenumbers >= lo) & (wavenumbers <= hi)
    wn   = wavenumbers[mask]

    ax.plot(wn, low_mc[:, mask].mean(axis=0),  color="steelblue", lw=2, label=f"低含水率 (n={len(low_mc)})")
    ax.plot(wn, high_mc[:, mask].mean(axis=0), color="coral",     lw=2, label=f"高含水率 (n={len(high_mc)})")
    ax.fill_between(wn, low_mc[:, mask].mean(axis=0),
                        high_mc[:, mask].mean(axis=0), alpha=0.15, color="gray")
    ax.set_title(f"{band_name}", fontsize=12)
    ax.set_xlabel("波数 (cm⁻¹)", fontsize=10)
    ax.set_ylabel("SNV後吸光度", fontsize=10)
    ax.invert_xaxis()  # NIR慣習：高波数→低波数
    ax.legend(fontsize=9)

plt.suptitle("水ピーク帯域別：低含水率 vs 高含水率の平均スペクトル (SNV後)", fontsize=13)
plt.tight_layout()
plt.show()

print("→ 帯域内で低含水率と高含水率のスペクトルに差があれば、その帯域の特徴量が有効")

In [ ]:
def extract_band_features(X, wavenumbers, band_lo, band_hi):
    """
    1つの帯域内からサンプルごとに特徴量を計算する。
    
    Returns
    -------
    dict of np.ndarray (各キーが特徴量名、値はサンプル数の配列)
    """
    mask = (wavenumbers >= band_lo) & (wavenumbers <= band_hi)
    wn   = wavenumbers[mask]
    X_b  = X[:, mask]         # (n_samples, n_wavenums_in_band)

    feats = {}
    feats["max"]    = X_b.max(axis=1)
    feats["min"]    = X_b.min(axis=1)
    feats["range"]  = feats["max"] - feats["min"]     # ← 吸光度の最大-最小の差
    feats["mean"]   = X_b.mean(axis=1)
    feats["std"]    = X_b.std(axis=1)
    feats["area"]   = np.trapz(X_b, x=wn, axis=1)    # 台形積分（面積）

    # ピーク位置（最大吸光度の波数）
    peak_idx        = X_b.argmax(axis=1)
    feats["pos"]    = wn[peak_idx]

    return feats


def extract_doublet_features(X, wavenumbers, band_lo=5000, band_hi=5400):
    """
    5200 cm⁻¹ 帯の2山（ダブレット）構造を検出して特徴量化する。
    
    2山が検出されない場合はNaNを返す（→後でfillna or 削除）。
    """
    mask = (wavenumbers >= band_lo) & (wavenumbers <= band_hi)
    wn   = wavenumbers[mask]
    X_b  = X[:, mask]
    n    = X_b.shape[0]

    doublet_gap    = np.full(n, np.nan)
    doublet_ratio  = np.full(n, np.nan)
    doublet_valley = np.full(n, np.nan)

    for i in range(n):
        spectrum = X_b[i]
        # 局所ピーク検出（幅3点以上、高さは帯域内std以上）
        peaks, props = find_peaks(spectrum, distance=3, prominence=spectrum.std() * 0.3)

        if len(peaks) >= 2:
            # 最も高い2つのピークを選ぶ
            top2     = sorted(peaks, key=lambda p: spectrum[p], reverse=True)[:2]
            top2     = sorted(top2)          # 波数の昇順に並べ直す
            p1, p2   = top2

            doublet_gap[i]    = abs(wn[p2] - wn[p1])
            doublet_ratio[i]  = spectrum[p1] / (spectrum[p2] + 1e-8)
            valley_idx        = np.argmin(spectrum[p1:p2]) + p1
            doublet_valley[i] = (spectrum[p1] + spectrum[p2]) / 2 - spectrum[valley_idx]

    return {
        "doublet_gap":    doublet_gap,
        "doublet_ratio":  doublet_ratio,
        "doublet_valley": doublet_valley,
    }


def build_handcrafted_features(X, wavenumbers):
    """
    全帯域の手作り特徴量を結合してDataFrameにして返す。
    SNV後のスペクトルを入力として想定。
    """
    all_feats = {}

    # --- A. 各水ピーク帯域の統計量 ---
    band_defs = {
        "8500": (8300, 8700),
        "6900": (6700, 7100),
        "5200": (5000, 5400),
    }
    for band_name, (lo, hi) in band_defs.items():
        feats = extract_band_features(X, wavenumbers, lo, hi)
        for feat_name, values in feats.items():
            all_feats[f"{feat_name}_{band_name}"] = values

    # --- B. ピーク間の比率・距離 ---
    area_8500 = all_feats["area_8500"]
    area_6900 = all_feats["area_6900"]
    area_5200 = all_feats["area_5200"]
    pos_6900  = all_feats["pos_6900"]
    pos_5200  = all_feats["pos_5200"]

    all_feats["ratio_8500_6900"]    = area_8500 / (area_6900 + 1e-8)
    all_feats["ratio_6900_5200"]    = area_6900 / (area_5200 + 1e-8)
    all_feats["pos_diff_6900_5200"] = pos_6900 - pos_5200    # ← 2ピーク間の波数距離

    # --- C. 5200帯のダブレット ---
    doublet = extract_doublet_features(X, wavenumbers)
    all_feats.update(doublet)

    # --- D. 全体形状 ---
    all_feats["global_range"]     = X.max(axis=1) - X.min(axis=1)
    hi_mask = wavenumbers > 7000
    lo_mask = wavenumbers < 6000
    all_feats["slope_high_low"]   = X[:, hi_mask].mean(axis=1) - X[:, lo_mask].mean(axis=1)

    return pd.DataFrame(all_feats)


# 実行
feat_train = build_handcrafted_features(X_snv_train, wavenumbers)
feat_test  = build_handcrafted_features(X_snv_test,  wavenumbers)

print(f"手作り特徴量数: {feat_train.shape[1]}")
print(feat_train.head(3))
print("\nNaN数:")
print(feat_train.isna().sum()[feat_train.isna().sum() > 0])

In [ ]:
# --- 特徴量と含水率の相関を確認する ---
# 相関が高い特徴量ほどモデルに効く可能性が高い

corr = feat_train.corrwith(pd.Series(y_train, name="含水率")).abs().sort_values(ascending=False)
print("含水率との絶対相関（上位）:")
print(corr.head(15).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
corr.head(15).plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("手作り特徴量と含水率の絶対相関（上位15）", fontsize=12)
ax.set_ylabel("|correlation|")
ax.set_xticklabels(corr.head(15).index, rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# --- 手作り特徴量 × LightGBM でCV評価 ---
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

# NaNをmedianで埋める（ダブレット未検出サンプル対応）
feat_train_filled = feat_train.fillna(feat_train.median())
feat_test_filled  = feat_test.fillna(feat_train.median())   # ← testはtrainのmedianで埋める

X_feat = feat_train_filled.values
X_feat_test = feat_test_filled.values

groups_sp = train["樹種"].values

# ベイスギ除外
mask_no_baisugi = groups_sp != "ベイスギ"
X_feat_cv   = X_feat[mask_no_baisugi]
y_cv        = y_train[mask_no_baisugi]
groups_cv   = groups_sp[mask_no_baisugi]

lgb_params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_child_samples": 10,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "random_state": 42,
}

gkf = GroupKFold(n_splits=5)
oof  = np.zeros(len(X_feat_cv))
test_pred = np.zeros(len(X_feat_test))

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_feat_cv, y_cv, groups_cv)):
    dtrain = lgb.Dataset(X_feat_cv[tr_idx], label=y_cv[tr_idx])
    dval   = lgb.Dataset(X_feat_cv[val_idx], label=y_cv[val_idx], reference=dtrain)
    model  = lgb.train(
        lgb_params, dtrain, num_boost_round=1000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(500)],
    )
    oof[val_idx] = model.predict(X_feat_cv[val_idx])
    test_pred += model.predict(X_feat_test) / 5
    rmse = mean_squared_error(y_cv[val_idx], oof[val_idx], squared=False)
    val_sp = np.unique(groups_cv[val_idx])
    print(f"Fold {fold+1} {val_sp}  RMSE={rmse:.4f}")

oof_rmse = mean_squared_error(y_cv, oof, squared=False)
print(f"\n=== OOF RMSE (手作り特徴量 × LightGBM): {oof_rmse:.4f} ===")
print("（参考: v2 LightGBM+PCA50 = 18.092）")

In [ ]:
# --- Feature Importance で効いている特徴量を確認 ---
# （最後のfoldのmodelで代用）

importances = model.feature_importance(importance_type="gain")
feat_names  = feat_train_filled.columns.tolist()

imp_df = pd.DataFrame({
    "feature": feat_names,
    "importance": importances
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
imp_df.head(15).plot(kind="bar", x="feature", y="importance", ax=ax,
                     color="coral", legend=False)
ax.set_title("Feature Importance（Gain）上位15", fontsize=12)
ax.set_ylabel("Importance")
ax.set_xticklabels(imp_df.head(15)["feature"], rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("\n上位特徴量:")
print(imp_df.head(10).to_string(index=False))

---

## 実装テンプレート

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter

# --- データ読み込み ---
train = pd.read_csv("../data/raw/train.csv", encoding="shift-jis")
test  = pd.read_csv("../data/raw/test.csv",  encoding="shift-jis")

META_COLS   = ["sample number", "species number", "樹種", "含水率"]
wave_cols   = [c for c in train.columns if c not in META_COLS]
wavenumbers = np.array(wave_cols, dtype=float)

X_train_raw = train[wave_cols].values.astype(float)
y_train     = train["含水率"].values
groups      = train["樹種"].values

X_test_raw  = test[wave_cols].values.astype(float)

print(f"train shape: {X_train_raw.shape}, test shape: {X_test_raw.shape}")

In [ ]:
# --- 前処理関数 ---

def snv(X):
    """Standard Normal Variate: 行単位の標準化"""
    mean = X.mean(axis=1, keepdims=True)
    std  = X.std(axis=1, keepdims=True)
    return (X - mean) / (std + 1e-8)

def sg_derivative(X, window=11, poly=2, deriv=2):
    """Savitzky-Golay 2次微分"""
    return np.apply_along_axis(
        lambda x: savgol_filter(x, window_length=window, polyorder=poly, deriv=deriv),
        axis=1, arr=X
    )

def msc(X_train, X_new):
    """
    MSC: trainの平均スペクトルを基準に補正
    X_new は1サンプルでも機能する（ルール上OK）
    """
    ref = X_train.mean(axis=0)
    result = np.zeros_like(X_new)
    for i in range(len(X_new)):
        coef = np.polyfit(ref, X_new[i], 1)
        result[i] = (X_new[i] - coef[1]) / coef[0]
    return result, ref

def select_water_bands(X, wavenumbers):
    """水の吸収帯のみを抽出"""
    bands = [
        (8300, 8700),  # OHの第2倍音
        (6700, 7100),  # OHの第1倍音
        (5000, 5400),  # OHの結合音
    ]
    mask = np.zeros(len(wavenumbers), dtype=bool)
    for lo, hi in bands:
        mask |= ((wavenumbers >= lo) & (wavenumbers <= hi))
    return X[:, mask], wavenumbers[mask]

print("前処理関数を定義しました")

In [ ]:
# --- CV評価の雛形（GroupKFold・樹種単位）---

def cv_evaluate(X, y, groups, model_fn, n_splits=5, exclude_species=None):
    """
    Parameters
    ----------
    X : np.ndarray  前処理済み特徴量
    y : np.ndarray  目的変数
    groups : np.ndarray  樹種ラベル（GroupKFold用）
    model_fn : callable  モデルを返す関数（引数なし）
    exclude_species : list  CVから除外する樹種名
    """
    if exclude_species:
        mask = ~np.isin(groups, exclude_species)
        X, y, groups = X[mask], y[mask], groups[mask]

    gkf = GroupKFold(n_splits=n_splits)
    oof_preds = np.zeros(len(y))
    fold_rmses = []

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        model = model_fn()
        model.fit(X_tr, y_tr)
        pred = model.predict(X_val).ravel()

        oof_preds[val_idx] = pred
        rmse = mean_squared_error(y_val, pred, squared=False)
        fold_rmses.append(rmse)

        val_species = np.unique(groups[val_idx])
        print(f"  Fold {fold+1}  val_species={val_species}  RMSE={rmse:.4f}")

    oof_rmse = mean_squared_error(y, oof_preds, squared=False)
    print(f"\n  OOF RMSE: {oof_rmse:.4f}")
    return oof_preds, oof_rmse, fold_rmses

print("CV評価関数を定義しました")

In [ ]:
# --- 優先実施 #1: PLSRの再チューニング ---
# SNV後の生スペクトルをそのままPLSRに投入

X_snv = snv(X_train_raw)

best_n, best_rmse = None, 1e9
results = []

for n_comp in [5, 8, 10, 15, 20, 25, 30]:
    print(f"\n--- n_components={n_comp} ---")
    _, rmse, _ = cv_evaluate(
        X_snv, y_train, groups,
        model_fn=lambda nc=n_comp: PLSRegression(n_components=nc),
        n_splits=5,
        exclude_species=["ベイスギ"]
    )
    results.append((n_comp, rmse))
    if rmse < best_rmse:
        best_rmse, best_n = rmse, n_comp

print(f"\nベスト n_components={best_n}  OOF RMSE={best_rmse:.4f}")

In [ ]:
# --- 優先実施 #2: アンサンブル ---
# v2のLightGBM予測値（submission_lgbm_v2.csv）とPLSR予測をブレンド
# ここでは仮の変数として示す

# pred_lgbm = ...  # v2のtest予測値（shape: (n_test,)）
# pred_plsr = ...  # PLSR再チューニング後のtest予測値

# for alpha in np.arange(0.5, 1.0, 0.1):
#     pred_ensemble = alpha * pred_lgbm + (1 - alpha) * pred_plsr
#     # CVで評価 → 最適αを選んでsubmit
#     print(f"alpha={alpha:.1f}  ...")

print("アンサンブルのテンプレートを確認しました")

In [ ]:
# --- 優先実施 #3: 水ピーク帯域絞り込み ---

X_snv_all = snv(X_train_raw)
X_water, wave_water = select_water_bands(X_snv_all, wavenumbers)
print(f"水ピーク帯域に絞った波数数: {X_water.shape[1]}")

# PLSRで評価
print("\n--- 水ピーク帯域 × PLSR ---")
_, rmse_water, _ = cv_evaluate(
    X_water, y_train, groups,
    model_fn=lambda: PLSRegression(n_components=10),
    n_splits=5,
    exclude_species=["ベイスギ"]
)

In [ ]:
# --- 優先実施 #4: SGのwindowチューニング ---

X_snv_all = snv(X_train_raw)

for window in [5, 7, 11, 15, 21]:
    try:
        X_sg = sg_derivative(X_snv_all, window=window, poly=2, deriv=2)
        print(f"\n--- SG window={window} ---")
        _, rmse, _ = cv_evaluate(
            X_sg, y_train, groups,
            model_fn=lambda: PLSRegression(n_components=10),
            n_splits=5,
            exclude_species=["ベイスギ"]
        )
    except Exception as e:
        print(f"window={window}: エラー ({e})")

In [ ]:
# --- submission生成テンプレート ---

def make_submission(test_df, pred, out_path):
    sub = test_df[["sample number"]].copy()
    sub["含水率"] = pred
    sub.to_csv(out_path, index=False, header=False, encoding="shift-jis")
    print(f"submission saved: {out_path}")
    return sub

# 使用例
# make_submission(test, pred_final, "../data/processed/submission_v4.csv")